# Discrete Flow Matching for 32×32 Mazes

This notebook implements one categorical discrete flow matching (DFM) model. Pixels remain binary categorical states throughout the forward and reverse processes. The source is an independent uniform Bernoulli grid; the target is a synthetic maze. A linear convex probability path connects them, and a timestep-conditioned U-Net predicts the endpoint posterior $p(x_1 \mid x_t,t)$.

This is one simple DFM parameterization, not the only one. The categorical path and posterior parameterization follow the framework in [Discrete Flow Matching (NeurIPS 2024)](https://proceedings.neurips.cc/paper_files/paper/f0d629a734b56a642701bba7bc8bb3ed-Paper-Conference.pdf).

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import importlib
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from plotly.subplots import make_subplots
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

from _shared.maze_data import generate_dataset, solvability_rate
import _shared.model as model_module

importlib.reload(model_module)
from _shared.model import Denoiser

SEED = 17
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project: {PROJECT_ROOT} | device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA runtime: {torch.version.cuda}")
else:
    print(
        f"CUDA unavailable to PyTorch (torch.version.cuda={torch.version.cuda}). Run uv sync and select the project .venv kernel."
    )

Project: c:\Users\shich\Src\discrete_diffusion | device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU | CUDA runtime: 12.8


## 1. Synthetic Dataset

Generate binary 32×32 perfect mazes and split them into train and validation sets. Pixel value 0 is a wall and 1 is a passage.

In [2]:
N_MAZES = 12_000
VAL_FRACTION = 0.1
mazes = generate_dataset(N_MAZES, seed=SEED)
n_val = int(N_MAZES * VAL_FRACTION)
train_mazes = torch.from_numpy(mazes[:-n_val].copy()).long()
val_mazes = torch.from_numpy(mazes[-n_val:].copy()).long()
print(
    "all:", mazes.shape, "| train:", len(train_mazes), "| validation:", len(val_mazes)
)
print(f"Generated-maze solvability: {solvability_rate(mazes):.1%}")

fig = go.Figure(
    go.Heatmap(
        z=mazes[0],
        zmin=0,
        zmax=1,
        colorscale=[[0, "black"], [1, "white"]],
        showscale=False,
    )
)
fig.update_layout(
    title="Synthetic maze example",
    width=420,
    height=420,
    yaxis=dict(autorange="reversed", scaleanchor="x"),
)
fig.show()

all: (12000, 32, 32) | train: 10800 | validation: 1200
Generated-maze solvability: 100.0%


## 2. Categorical Convex Probability Path

For paired source $x_0$ and data maze $x_1$, each pixel follows
$p_t(x_i\mid x_0,x_1)=(1-t)\delta_{x_0}+t\delta_{x_1}$, with $t\in[0,1]$.

To draw $x_t$, choose the source class with probability $1-t$ and the target class with probability $t$. Each state remains exactly 0 or 1: this is not a grayscale interpolation. The preview uses the equivalent jump process: a pixel that has not reached its target jumps there with probability $\Delta t/(1-t)$.

In [3]:
NUM_CLASSES = 2
FLOW_STEPS = 100


def sample_conditional_path(x0, x1, times):
    """Draw categorical x_t from the linear source-to-target path."""
    t = times.to(device=x0.device, dtype=torch.float32)
    choose_target = torch.rand(x0.shape, device=x0.device) < t[:, None, None]
    return torch.where(choose_target, x1, x0)


def one_hot_pixels(x):
    """Convert integer (B,H,W) labels into (B,2,H,W) categorical channels."""
    return F.one_hot(x.long(), num_classes=NUM_CLASSES).permute(0, 3, 1, 2).float()


preview_target = torch.from_numpy(mazes[0].copy()).long().to(DEVICE)
preview_source = torch.randint(0, NUM_CLASSES, preview_target.shape, device=DEVICE)
preview_frames = [preview_source.clone()]
current = preview_source.clone()
for step in range(FLOW_STEPS):
    t = step / FLOW_STEPS
    jump_probability = min((1.0 / FLOW_STEPS) / max(1.0 - t, 1e-8), 1.0)
    jump = (torch.rand(current.shape, device=DEVICE) < jump_probability) & (
        current != preview_target
    )
    current = torch.where(jump, preview_target, current)
    if (step + 1) % 10 == 0:
        preview_frames.append(current.clone())

preview_times = np.linspace(0.0, 1.0, 11)
fig = make_subplots(
    rows=2, cols=6, subplot_titles=[f"t={t:.1f}" for t in preview_times]
)
for index, frame in enumerate(preview_frames):
    fig.add_trace(
        go.Heatmap(
            z=frame.cpu().numpy(),
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=index // 6 + 1,
        col=index % 6 + 1,
    )
fig.update_layout(
    title="Conditional DFM path: categorical uniform source to maze",
    width=1500,
    height=560,
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange="reversed")
fig.show()
print("Source class-1 fraction:", f"{preview_source.float().mean().item():.3f}")
print("Final state matches target:", bool(torch.equal(current, preview_target)))

Source class-1 fraction: 0.493
Final state matches target: True


## 3. Model Architecture

The shared timestep-conditioned U-Net consumes two one-hot categorical channels and returns two per-pixel logits for the endpoint posterior $p(x_1\mid x_t,t)$. Time is continuous in [0,1] and scaled to [0,100] before the shared sinusoidal time embedding.

In [4]:
BASE_CHANNELS = 64
TIME_DIM = 128
TIME_EMBEDDING_SCALE = 100.0

model = Denoiser(
    embedding_dim=NUM_CLASSES,
    output_dim=NUM_CLASSES,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
).to(DEVICE)
print(model)
print(
    f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

Denoiser(
  (time_mlp): Sequential(
    (0): Linear(in_features=128, out_features=512, bias=True)
    (1): SiLU()
    (2): Linear(in_features=512, out_features=128, bias=True)
  )
  (input): Conv2d(2, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc32): ResBlock(
    (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_projection): Linear(in_features=128, out_features=64, bias=True)
    (norm2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (skip): Identity()
  )
  (down16): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (enc16): ResBlock(
    (norm1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_projection): Linear(in_features=128, out_features=128, bias=True)
    (norm2): GroupNorm(8, 128, eps=1e-05, a

## 4. Discrete Flow Matching Training

Sample a uniform categorical source $x_0$, pair it with a training maze $x_1$, draw $x_t$ from the convex path, and train with pixelwise cross-entropy to predict $x_1$ from $(x_t,t)$. This endpoint posterior gives the categorical jump rates at sampling time.

For binary pixels, the learned rate from the current class to the other class is
$u_t(\mathrm{flip}\mid x_t)=P_\theta(x_1\ne x_t\mid x_t,t)/(1-t)$.
Over a finite interval, use the corresponding integrated path hazard:
$P_\theta(x_1\ne x_t\mid x_t,t)\,\Delta t/(1-t)$.

In [5]:
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 2e-4
PLOT_EVERY = 100

train_loader = DataLoader(
    TensorDataset(train_mazes), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)
val_loader = DataLoader(TensorDataset(val_mazes), batch_size=BATCH_SIZE, shuffle=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


def batch_loss(maze_batch):
    x1 = maze_batch.to(DEVICE)
    x0 = torch.randint(0, NUM_CLASSES, x1.shape, device=DEVICE)
    times = torch.rand(x1.shape[0], device=DEVICE)
    xt = sample_conditional_path(x0, x1, times)
    logits_x1 = model(one_hot_pixels(xt), times * TIME_EMBEDDING_SCALE)
    return F.cross_entropy(logits_x1, x1)


@torch.no_grad()
def validation_loss():
    model.eval()
    values = [batch_loss(batch).item() for (batch,) in val_loader]
    model.train()
    return float(np.mean(values))


model.train()
train_history, val_history = [], []
loss_fig = go.Figure()
loss_fig.add_scatter(name="train", mode="lines")
loss_fig.add_scatter(name="validation", mode="lines+markers")
loss_fig.update_layout(
    title="Discrete flow matching loss",
    xaxis_title="Optimization step",
    yaxis_title="Pixel cross-entropy",
    yaxis_type="log",
)
display(loss_fig)

global_step = 0
for epoch in range(EPOCHS):
    epoch_losses = []
    for (batch,) in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = batch_loss(batch)
        loss.backward()
        optimizer.step()
        train_history.append(loss.item())
        epoch_losses.append(loss.item())
        global_step += 1
        if global_step % PLOT_EVERY == 0:
            loss_fig.data[0].x = list(range(1, len(train_history) + 1))
            loss_fig.data[0].y = train_history
            loss_fig.data[1].x = [step for step, _ in val_history]
            loss_fig.data[1].y = [value for _, value in val_history]
            loss_fig.update_layout(
                title=f"Discrete flow matching | epoch {epoch + 1}/{EPOCHS} | step {global_step}"
            )
            clear_output(wait=True)
            display(loss_fig)
    epoch_validation = validation_loss()
    val_history.append((global_step, epoch_validation))
    loss_fig.data[0].x = list(range(1, len(train_history) + 1))
    loss_fig.data[0].y = train_history
    loss_fig.data[1].x = [step for step, _ in val_history]
    loss_fig.data[1].y = [value for _, value in val_history]
    loss_fig.update_layout(
        title=f"Discrete flow matching | epoch {epoch + 1}/{EPOCHS} | step {global_step}"
    )
    clear_output(wait=True)
    display(loss_fig)
    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | train={np.mean(epoch_losses):.4f} | val={epoch_validation:.4f}"
    )

print(
    f"Training complete | train={np.mean(epoch_losses):.4f} | val={epoch_validation:.4f}"
)

Epoch 30/30 | train=0.1842 | val=0.1834
Training complete | train=0.1842 | val=0.1834


## 5. Save the Model

Save model weights and the path, prediction, and architecture settings to the ignored artifacts directory.

In [6]:
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
checkpoint_path = ARTIFACTS / "maze_discrete_flow_matching.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": {
            "num_classes": NUM_CLASSES,
            "flow_steps": FLOW_STEPS,
            "source_distribution": "uniform_independent_categorical",
            "probability_path": "linear_convex_categorical",
            "prediction_target": "x1_posterior_logits",
            "base_channels": BASE_CHANNELS,
            "time_dim": TIME_DIM,
            "time_embedding_scale": TIME_EMBEDDING_SCALE,
        },
    },
    checkpoint_path,
)
print(f"Saved: {checkpoint_path}")

Saved: c:\Users\shich\Src\discrete_diffusion\artifacts\maze_discrete_flow_matching.pt


## 6. Reverse Sampling and Solvability

Start from a fresh uniform binary grid at $t=0$. At each step evaluate the endpoint posterior and independently flip pixels according to the learned discrete flow probability. The state remains categorical throughout; there is no embedding projection or temperature schedule. Finally measure connectivity between the fixed maze entrance and exit.

In [7]:
N_METRIC_SAMPLES = 128
N_TO_SHOW = 4
PREVIEW_TIMES = list(range(0, FLOW_STEPS + 1, 10))


@torch.no_grad()
def sample_discrete_flow(model, n_samples, capture_times=()):
    model.eval()
    x = torch.randint(0, NUM_CLASSES, (n_samples, 32, 32), device=DEVICE)
    initial_prior = x.clone()
    captures = {0: x.clone()} if 0 in capture_times else {}
    capture_times = set(capture_times)

    for step in range(FLOW_STEPS):
        t = step / FLOW_STEPS
        dt = 1.0 / FLOW_STEPS
        times = torch.full((n_samples,), t * TIME_EMBEDDING_SCALE, device=DEVICE)
        endpoint_probabilities = model(one_hot_pixels(x), times).softmax(dim=1)
        p_current = endpoint_probabilities.gather(1, x[:, None]).squeeze(1)
        p_endpoint_differs = (1.0 - p_current).clamp(0.0, 1.0)
        path_hazard_fraction = min(dt / max(1.0 - t, 1e-8), 1.0)
        p_jump = (p_endpoint_differs * path_hazard_fraction).clamp(0.0, 1.0)
        jump = torch.rand(p_jump.shape, device=DEVICE) < p_jump
        x = torch.where(jump, 1 - x, x)
        if step + 1 in capture_times:
            captures[step + 1] = x.clone()
    return x, captures, initial_prior


samples, snapshots, initial_prior = sample_discrete_flow(
    model, N_METRIC_SAMPLES, PREVIEW_TIMES
)
hard_samples = samples.cpu().numpy().astype(np.uint8)
print(
    f"Initial uniform prior class-1 fraction: {initial_prior.float().mean().item():.3f} (expected about 0.5)"
)

fig = make_subplots(
    rows=N_TO_SHOW,
    cols=len(PREVIEW_TIMES),
    subplot_titles=[f"t={step}" for _ in range(N_TO_SHOW) for step in PREVIEW_TIMES],
)
for row in range(N_TO_SHOW):
    for col, step in enumerate(PREVIEW_TIMES, start=1):
        fig.add_trace(
            go.Heatmap(
                z=snapshots[step][row].cpu().numpy(),
                zmin=0,
                zmax=1,
                colorscale=[[0, "black"], [1, "white"]],
                showscale=False,
            ),
            row=row + 1,
            col=col,
        )
fig.update_layout(
    title="Reverse discrete flow sampling: categorical state over time",
    height=820,
    width=1900,
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange="reversed")
fig.show()

solved = np.array([solvability_rate([maze]) for maze in hard_samples], dtype=bool)
print(f"Solvable samples: {solved.sum()}/{len(solved)} ({solved.mean():.1%})")
hard_fig = make_subplots(
    rows=1,
    cols=N_TO_SHOW,
    subplot_titles=[
        f"Sample {i + 1}: {'solvable' if solved[i] else 'unsolvable'}"
        for i in range(N_TO_SHOW)
    ],
)
for i in range(N_TO_SHOW):
    hard_fig.add_trace(
        go.Heatmap(
            z=hard_samples[i],
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=1,
        col=i + 1,
    )
hard_fig.update_layout(title="Final discrete flow maze samples", height=390, width=900)
hard_fig.update_xaxes(showticklabels=False)
hard_fig.update_yaxes(showticklabels=False, autorange="reversed")
hard_fig.show()

Initial uniform prior class-1 fraction: 0.499 (expected about 0.5)


Solvable samples: 50/128 (39.1%)
